## Оценка рыночной стоимости недвижимости

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from datetime import datetime, timezone

### Первичный анализ

In [2]:
df = pd.read_csv('USA_Housing_Dataset.csv')
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-09 00:00:00,376000.0,3.0,2.00,1340,1384,3.0,0,0,3,1340,0,2008,0,9245-9249 Fremont Ave N,Seattle,WA 98103,USA
1,2014-05-09 00:00:00,800000.0,4.0,3.25,3540,159430,2.0,0,0,3,3540,0,2007,0,33001 NE 24th St,Carnation,WA 98014,USA
2,2014-05-09 00:00:00,2238888.0,5.0,6.50,7270,130017,2.0,0,0,3,6420,850,2010,0,7070 270th Pl SE,Issaquah,WA 98029,USA
3,2014-05-09 00:00:00,324000.0,3.0,2.25,998,904,2.0,0,0,3,798,200,2007,0,820 NW 95th St,Seattle,WA 98117,USA
4,2014-05-10 00:00:00,549900.0,5.0,2.75,3060,7015,1.0,0,0,5,1600,1460,1979,0,10834 31st Ave SW,Seattle,WA 98146,USA


In [3]:
print(f'--- Shape ---\n{df.shape}\n')
print(f'--- Features ---\n{df.columns}\n')
print(f'--- Feature Types ---\n{df.dtypes}\n')

--- Shape ---
(4140, 18)

--- Features ---
Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country'],
      dtype='str')

--- Feature Types ---
date                 str
price            float64
bedrooms         float64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
street               str
city                 str
statezip             str
country              str
dtype: object



In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
price,4140.0,553062.877289,583686.452245,0.0,320000.00,460000.00,659125.0,26590000.00
bedrooms,4140.0,3.400483,0.903939,0.0,3.00,3.00,4.0,8.00
bathrooms,4140.0,2.163043,0.784733,0.0,1.75,2.25,2.5,6.75
sqft_living,4140.0,2143.638889,957.481621,370.0,1470.00,1980.00,2620.0,10040.00
sqft_lot,4140.0,14697.638164,35876.838123,638.0,5000.00,7676.00,11000.0,1074218.00
floors,4140.0,1.514130,0.534941,1.0,1.00,1.50,2.0,3.50
waterfront,4140.0,0.007488,0.086219,0.0,0.00,0.00,0.0,1.00
view,4140.0,0.246618,0.790619,0.0,0.00,0.00,0.0,4.00
condition,4140.0,3.452415,0.678533,1.0,3.00,3.00,4.0,5.00
sqft_above,4140.0,1831.351449,861.382947,370.0,1190.00,1600.00,2310.0,8020.00


In [5]:
df.isna().sum() # кол-во пропущенных значений

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
street           0
city             0
statezip         0
country          0
dtype: int64

In [6]:
df.duplicated().sum() # кол-во дубликатов

np.int64(0)

In [7]:
df = df.drop_duplicates()

### Feature Engineering

In [8]:
df.index

RangeIndex(start=0, stop=4140, step=1)

In [9]:
df[df['price'] == 0].head(3) # некорректные значения в целевой переменной price

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
3894,2014-05-05 00:00:00,0.0,3.0,1.75,1490,10125,1.0,0,0,4,1490,0,1962,0,3911 S 328th St,Federal Way,WA 98001,USA
3896,2014-05-05 00:00:00,0.0,4.0,2.75,2600,5390,1.0,0,0,4,1300,1300,1960,2001,2120 31st Ave W,Seattle,WA 98199,USA
3897,2014-05-05 00:00:00,0.0,6.0,2.75,3200,9200,1.0,0,2,4,1600,1600,1953,1983,12271 Marine View Dr SW,Burien,WA 98146,USA


### Обучение модели

In [10]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [11]:
def transform_data(data):
    current_year = datetime.now(timezone.utc).year
    data['zip'] = data['statezip'].str.extract(r'(\d+)').astype(str)
    data['house_age'] = current_year - data['yr_built']
    data['was_renovated'] = (data['yr_renovated'] > 0).astype(int)
    data['total_rooms'] = data['bedrooms'] + data['bathrooms']
    data['sqft_per_room'] = data['sqft_living'] / (data['total_rooms'] + 1)
    data['lot_utilization'] = data['sqft_living'] / (data['sqft_lot'] + 1)
    zip_price_map = data.groupby('zip')['price'].median()
    data['zip_price_level'] = data['zip'].map(zip_price_map)
    return data

In [12]:
df = pd.read_csv('USA_Housing_Dataset.csv')

q_low = df['price'].quantile(0.01)
q_high = df['price'].quantile(0.99)
df = df[(df['price'] > q_low) & (df['price'] < q_high)].copy()
df = df.reset_index(drop=True)

df = transform_data(df)

drop_cols = ['price', 'date', 'street', 'statezip', 'country', 'yr_built', 'yr_renovated', 'sqft_above']
X = df.drop(columns=drop_cols)
y = np.log1p(df['price'])

cat_features = ['city', 'zip', 'waterfront', 'view', 'condition', 'was_renovated']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.02,
    depth=7,
    l2_leaf_reg=12,
    random_seed=42,
    cat_features=cat_features,
    early_stopping_rounds=100,
    loss_function='RMSE',
    verbose=0
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

print(f"MAE: {mean_absolute_error(y_true, y_pred):.2f}")
print(f"MSE: {mean_squared_error(y_true, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.2f}")
print(f"R2 Score: {r2_score(y_true, y_pred):.4f}")

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.get_feature_importance()
}).sort_values(by='importance', ascending=False)

print(feature_importance.head(7))

MAE: 80652.87
MSE: 19245529850.27
RMSE: 138728.26
R2 Score: 0.7815
            feature  importance
16  zip_price_level   38.338035
2       sqft_living   23.579273
9              city    7.546153
6              view    5.094194
14    sqft_per_room    4.985903
1         bathrooms    4.223398
10              zip    2.825283


### Сохранение модели

In [13]:
model.save_model("../ml/artifacts/model.cbm")

In [ ]:
from minio import Minio
from minio.error import S3Error


minio_client = Minio(
    endpoint='localhost:9000',
    access_key='admin',
    secret_key='password',
    secure=False
)

bucket = 'artifacts'

model_src = '../ml/artifacts/model.cbm'
metadata_src = '../ml/artifacts/metadata.json'

model_filename = 'model.cbm'
metadata_filename = 'metadata.json'

try:
    if not minio_client.bucket_exists(bucket):
        minio_client.make_bucket(bucket)
    
    minio_client.fput_object(bucket, model_filename, model_src)
    minio_client.fput_object(bucket, metadata_filename, metadata_src)
except S3Error as err:
    print(err.message)


The bucket you tried to delete is not empty
